# 03 - EDA and Feature Engineering**Objective:** Explore income data patterns and create new features.**Steps:**1. Statistical summary and class distribution2. Visualizations (distributions, correlations, feature comparison by income)3. Feature scaling4. Feature engineering (interaction and ratio features)5. Save engineered data

In [ ]:
import pandas as pdimport numpy as npfrom pathlib import Pathimport matplotlib.pyplot as pltimport seaborn as snssns.set_style("whitegrid")plt.rcParams["figure.figsize"] = (10, 6)print("Libraries imported successfully")

In [ ]:
# TODO: Load the clean train data and inspect its structure# Start by reading data/processed/clean_train.csv into a DataFrame.# Then use .info() to see column dtypes and non-null counts,# .head() to preview a few rows, and .describe() for summary statistics.# The dataset has 101 columns: 6 numeric features + income target# + 94 one-hot encoded categorical columns.PROCESSED_DIR = Path("../data/processed")# df = pd.read_csv(PROCESSED_DIR / "clean_train.csv")# print(f"Shape: {df.shape}")# print(df.info())# print(df.head())# print(df.describe())

### Statistical Summary & DistributionBefore building any models, it is important to understand the range and spread of your data.The target variable — `income` — is binary (a classification problem).Check the class balance to see if the dataset is imbalanced, which might affect model evaluation.Income data has 6 original numeric features plus 94 one-hot encoded categorical indicators.Key numeric features: age, fnlwgt, education-num, capital-gain, capital-loss, hours-per-week.

In [ ]:
# TODO: Examine the target variable distribution and feature summary# Use df.describe() to see min, max, mean, and quartiles for all features.# Check the class balance with df["income"].value_counts().# The dataset is imbalanced: roughly 75% <=50K and 25% >50K.## print(df.describe())# print(df["income"].value_counts())# print(df["income"].value_counts(normalize=True))

#### A little primer on groupby - `groupby` is a powerful pandas method that allows you to split your data into groups based on some criteria, apply a function to each group, and then combine the results. For example, to see how average age differs between high and low income groups, you can do:```pythonage_by_income = df.groupby("income")["age"].mean()```- `aggregate` is a method that allows you to apply multiple functions to your grouped data. For example, to get both the mean and standard deviation of age by income, you can do:```pythonstats_by_income = df.groupby("income")["age"].aggregate(["mean", "std"])```Aggregate functions can be any function that takes a Series and returns a single value, such as `mean`, `std`, `min`, `max`, etc.Aggregate can be deployed on multiple columns at once, and you can specify different functions for each column if needed.

In [ ]:
# === Executed Example: GroupBy and Aggregate ===# Small inline dataset showing how groupby splits data by income# and compares demographics between classes.import pandas as pddata = pd.DataFrame({    "income": [0, 0, 0, 1, 1, 1],    "age": [25, 30, 28, 45, 50, 42],    "hours-per-week": [40, 35, 38, 55, 60, 50],})mean_by_income = data.groupby("income")["age"].mean()print("Average age by income:\n", mean_by_income)stats_by_income = data.groupby("income")["hours-per-week"].agg(["mean", "std", "min", "max"])print("\nHours-per-week statistics by income:\n", stats_by_income)

In [ ]:
# === Commented Template: GroupBy and Aggregate ===# Uncomment and adapt to your own dataset.# import pandas as pd# data = pd.DataFrame({#     "group_col": [val1, val1, val2, val2],#     "value_col": [10, 20, 30, 40],# })# grouped = data.groupby("group_col")["value_col"].mean()# stats = data.groupby("group_col")["value_col"].agg(["mean", "std", "min", "max"])

### Missing Value Imputation

Our clean data has no missing values, but the corrupted variants from
`data_injection/` do. Common strategies:

- **Drop rows**: `df.dropna()` — fast, loses samples
- **Mean/Median imputation**: `SimpleImputer(strategy='median')` — preserves sample count
- **KNN imputation**: `KNNImputer()` — estimates from neighbors, more robust
- **Forward fill**: `df.ffill()` — for sequential data

In [ ]:
# === Executed Example: Missing Value Imputation ===
# Using SimpleImputer to fill missing values with the mean.

from sklearn.impute import SimpleImputer

data = pd.DataFrame({
    "age": [25.0, np.nan, 48.0, 35.0, np.nan],
    "hours-per-week": [40.0, 50.0, np.nan, 30.0, 45.0],
})

print("Before imputation:")
print(data)

imputer = SimpleImputer(strategy='mean')
imputed = imputer.fit_transform(data)
imputed_df = pd.DataFrame(imputed, columns=data.columns)
print("\nAfter imputation (mean strategy):")
print(imputed_df)
print(f"\nImputed age: {imputer.statistics_[0]:.3f}")
print(f"Imputed hours-per-week: {imputer.statistics_[1]:.3f}")

In [ ]:
# TODO: Compare feature distributions by income group# Which numeric features best separate <=50K (0) from >50K (1)?# Use boxplots to compare key features.## features_to_plot = ["age", "hours-per-week", "capital-gain", "education-num"]# fig, axes = plt.subplots(2, 2, figsize=(12, 10))# for ax, feature in zip(axes.ravel(), features_to_plot):#     sns.boxplot(x="income", y=feature, data=df, ax=ax)#     ax.set_title(f"{feature} by income")# plt.tight_layout()# plt.show()

### VisualizationsVisual exploration helps you spot patterns and relationships that summary statistics miss.Focus on:- How each feature is distributed (histograms)- Which features correlate with the target (heatmap)- Which features separate high-income from low-income groups (boxplots)

In [ ]:
# TODO: Plot histograms for key numerical features# Pick the 6 original numeric features.# Use df[features].hist() with bins=30 and a large figure size.# features_to_plot = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]# df[features_to_plot].hist(bins=30, figsize=(15, 10))# plt.tight_layout()# plt.show()

In [ ]:
# TODO: Create a correlation heatmap# Use df.corr() to compute pairwise correlations between numeric columns.# With 101 columns, the heatmap will be dense. Pick the top features# most correlated with income, or limit to a subset of 15-20 columns.# Then visualize with sns.heatmap().## TODO: Identify the top features correlated with income# Sort the correlation values for the target column to see which features# have the strongest relationship with high income.# numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()# top_features = df[numeric_cols].corr()["income"].abs().sort_values(ascending=False).head(15).index# plt.figure(figsize=(12, 10))# sns.heatmap(df[top_features].corr(), annot=True, cmap="coolwarm", center=0, fmt=".2f")# plt.title("Top 15 Features Correlation Heatmap")# plt.show()## target_corr = df[numeric_cols].corr()["income"].sort_values(ascending=False)# print("Top correlations with income:")# print(target_corr.head(10))

### Feature ScalingMany machine learning algorithms (SVM, logistic regression, neural networks) are sensitive tothe scale of input features. StandardScaler transforms each feature to have mean 0 andstandard deviation 1, which puts all features on equal footing.Tree-based models (Random Forest, XGBoost) do not require scaling since they split onthresholds independently of feature magnitude.

In [ ]:
# === Executed Example: Feature Scaling ===# Small inline dataset showing how StandardScaler transforms features# to have mean ~0 and std ~1.from sklearn.preprocessing import StandardScalerimport pandas as pddata = pd.DataFrame({    "age": [25, 30, 28, 45, 50],    "hours-per-week": [40, 35, 38, 55, 60],    "income": [0, 0, 0, 1, 1],})scaler = StandardScaler()scaled_features = scaler.fit_transform(data[["age", "hours-per-week"]])scaled_df = pd.DataFrame(scaled_features, columns=["age_scaled", "hours_scaled"])scaled_df["income"] = data["income"]print(scaled_df)print(f"Means after scaling: {scaled_df[['age_scaled', 'hours_scaled']].mean().values}")print(f"Stds after scaling: {scaled_df[['age_scaled', 'hours_scaled']].std().values}")

In [ ]:
from sklearn.preprocessing import StandardScaler# TODO: Scale the numeric features# The dataset has 6 original numeric columns plus 94 one-hot encoded columns.# One-hot columns are already 0/1 so they don't need scaling.# Select the 6 numeric features and scale them.## Create a StandardScaler, call fit_transform() to compute the mean and std# and return the scaled array in one step.## After scaling, verify that each feature has mean ~0 and std ~1.# numeric_features = ["age", "fnlwgt", "education-num", "capital-gain", "capital-loss", "hours-per-week"]# scaler = StandardScaler()# df_scaled = scaler.fit_transform(df[numeric_features])## Rename the scaled columns with a "_scaled" suffix# df_scaled = pd.DataFrame(df_scaled, columns=[f"{col}_scaled" for col in numeric_features])# print(f"Means after scaling: {df_scaled.mean().values}")# print(f"Stds after scaling: {df_scaled.std().values}")# print(f"All means near zero: {np.allclose(df_scaled.mean(), 0, atol=1e-10)}")

### Feature EngineeringNew features derived from existing columns can capture interactions and non-linear relationships.Good candidates for income prediction:- **Capital total**: `capital-gain - capital-loss` — net investment income- **Hours × Education**: `hours-per-week * education-num` — combined work effort and skill level- **Age squared**: `age^2` — captures non-linear age effects on earning potentialBe careful with division by zero — add a small epsilon or +1 to the denominator.#### NoteIn pandas, you can create interaction features like this:```pythondf["feature1_feature2"] = df["feature1"] * df["feature2"]```

In [ ]:
# === Executed Example: Interaction Features ===# Multiplication and difference on a small inline dataset.import pandas as pddata = pd.DataFrame({    "capital-gain": [0, 5000, 0, 20000, 1000],    "capital-loss": [0, 0, 2000, 0, 500],    "hours-per-week": [40, 35, 38, 55, 60],    "education-num": [9, 13, 10, 14, 12],})data["capital_total"] = data["capital-gain"] - data["capital-loss"]print("Capital total:\n", data[["capital-gain", "capital-loss", "capital_total"]])data["hours_education"] = data["hours-per-week"] * data["education-num"]print("\nHours x Education:\n", data[["hours-per-week", "education-num", "hours_education"]])

In [ ]:
# === Commented Template: Interaction Features ===# Uncomment and adapt to your own dataset.# import pandas as pd# data = pd.DataFrame({#     "feature_a": [val1, val2, val3],#     "feature_b": [val1, val2, val3],# })# data["a_times_b"] = data["feature_a"] * data["feature_b"]# EPS = 1e-6# data["a_over_b"] = data["feature_a"] / (data["feature_b"] + EPS)

In [ ]:
# TODO: Create new features based on domain knowledge# The income dataset combines demographics, work history, and financial data.# Interaction features can capture combined effects on earning potential.## Examples:#   df["capital_total"] = df["capital-gain"] - df["capital-loss"]#   df["hours_education"] = df["hours-per-week"] * df["education-num"]#   df["age_squared"] = df["age"] ** 2## TODO: Verify the new features# Check that they have finite values and reasonable ranges with .describe().

In [ ]:
# TODO: Save the engineered data for the next notebook# Include both the original and new features.PROCESSED_DIR = Path("../data/processed")PROCESSED_DIR.mkdir(parents=True, exist_ok=True)# df.to_csv(PROCESSED_DIR / "engineered_train.csv", index=False)# print("Engineered data saved to data/processed/engineered_train.csv")

### Exercises1. **Try different scalers**: Replace `StandardScaler` with `MinMaxScaler` or `RobustScaler` and compare how the scaled distributions look.2. **Feature importance hint**: Use `df.groupby("income")[features].mean()` to find which one-hot encoded categorical features have the largest difference between income groups (e.g., `education_ Bachelors`, `occupation_ Exec-managerial`).3. **Create a capital ratio**: Compute `df["capital_ratio"] = df["capital-gain"] / (df["capital-loss"] + 1)`. Does a high ratio correspond to >50K income?4. **Log transform capital-gain**: The `capital-gain` and `capital-loss` features are heavily zero-inflated and right-skewed. Try `np.log1p()` on them and check if the distribution becomes more interpretable.5. **Pairplot**: Use `sns.pairplot()` on a subset of the most discriminative numeric features, coloring by income — do high and low income individuals form distinct clusters?